## Fine-tuning Problem & Setup

In [1]:
import sys
import platform
import torch

print("Python version:", sys.version)
print("Platform:", platform.platform())
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print("GPU: Not available")

Python version: 3.11.8 (tags/v3.11.8:db85d51, Feb  6 2024, 22:03:32) [MSC v.1937 64 bit (AMD64)]
Platform: Windows-10-10.0.22631-SP0
PyTorch version: 2.14.0+cpu
CUDA available: False
GPU: Not available


In [2]:
required_packages = [
    "transformers",
    "datasets",
    "peft",
    "accelerate",
    "bitsandbytes",
    "trl"
]

for package in required_packages:
    try:
        module = __import__(package)
        version = getattr(module, "__version__", "installed")
        print(f"{package}: {version}")
    except ImportError:
        print(f"{package}: NOT INSTALLED")

transformers: 5.17.0
datasets: NOT INSTALLED
peft: NOT INSTALLED
accelerate: NOT INSTALLED
bitsandbytes: NOT INSTALLED
trl: NOT INSTALLED


In [3]:
import pandas as pd

customer_features = pd.read_csv("../data/customer_features.csv")
product_features = pd.read_csv("../data/product_features.csv")
order_analytics = pd.read_csv("../data/order_analytics.csv")

print("Customer features:", customer_features.shape)
print("Product features:", product_features.shape)
print("Order analytics:", order_analytics.shape)

Customer features: (4335, 13)
Product features: (3918, 11)
Order analytics: (19865, 9)


In [4]:
print("Customer columns:")
print(customer_features.columns.tolist())

print("\nProduct columns:")
print(product_features.columns.tolist())

print("\nOrder columns:")
print(order_analytics.columns.tolist())

Customer columns:
['Customer ID', 'Total_Orders', 'Total_Revenue', 'Total_Units', 'Unique_Products', 'Recency_Days', 'Customer_Lifetime_Days', 'Average_Order_Value', 'Average_Units_Per_Order', 'Average_Products_Per_Order', 'Orders_Per_Month', 'Revenue_Segment', 'Activity_Segment']

Product columns:
['StockCode', 'Product_Name', 'Total_Units_Sold', 'Total_Revenue', 'Orders', 'Customers', 'Average_Price', 'Revenue_Per_Order', 'Units_Per_Order', 'Customer_Penetration', 'Revenue_Segment']

Order columns:
['Invoice', 'Order_Revenue', 'Units', 'Unique_Products', 'Average_Item_Value', 'Customer_ID', 'Country', 'Order_Date', 'Basket_Size']


## Instruction Dataset Creation

In [5]:
import pandas as pd
import json

instruction_data = []

for _, row in customer_features.iterrows():
    instruction_data.append({
        "instruction": (
            "Analyze this customer's purchasing behavior and provide "
            "a concise business interpretation."
        ),
        "input": (
            f"Total orders: {int(row['Total_Orders'])}; "
            f"Total revenue: £{row['Total_Revenue']:,.2f}; "
            f"Total units: {int(row['Total_Units']):,}; "
            f"Unique products: {int(row['Unique_Products'])}; "
            f"Recency: {int(row['Recency_Days'])} days; "
            f"Average order value: £{row['Average_Order_Value']:,.2f}; "
            f"Revenue segment: {row['Revenue_Segment']}; "
            f"Activity segment: {row['Activity_Segment']}."
        ),
        "output": (
            f"This customer belongs to the {row['Revenue_Segment']} "
            f"revenue segment and is classified as {row['Activity_Segment']}. "
            f"They have placed {int(row['Total_Orders'])} orders with "
            f"£{row['Total_Revenue']:,.2f} in total revenue. "
            f"The customer purchased {int(row['Unique_Products'])} unique "
            f"products and has a recency of {int(row['Recency_Days'])} days."
        )
    })

instruction_df = pd.DataFrame(instruction_data)

print("Instruction dataset shape:", instruction_df.shape)
print("\nColumns:")
print(instruction_df.columns.tolist())

Instruction dataset shape: (4335, 3)

Columns:
['instruction', 'input', 'output']


In [6]:
pd.set_option("display.max_colwidth", 500)

display(
    instruction_df[
        ["instruction", "input", "output"]
    ].head(3)
)

,instruction,input,output
0,Analyze this customer's purchasing behavior and provide a concise business interpretation.,"Total orders: 1; Total revenue: £77,183.60; Total units: 74,215; Unique products: 1; Recency: 326 days; Average order value: £77,183.60; Revenue segment: Very High; Activity segment: Inactive.","This customer belongs to the Very High revenue segment and is classified as Inactive. They have placed 1 orders with £77,183.60 in total revenue. The customer purchased 1 unique products and has a recency of 326 days."
1,Analyze this customer's purchasing behavior and provide a concise business interpretation.,"Total orders: 7; Total revenue: £4,310.00; Total units: 2,458; Unique products: 103; Recency: 2 days; Average order value: £615.71; Revenue segment: Very High; Activity segment: Active.","This customer belongs to the Very High revenue segment and is classified as Active. They have placed 7 orders with £4,310.00 in total revenue. The customer purchased 103 unique products and has a recency of 2 days."
2,Analyze this customer's purchasing behavior and provide a concise business interpretation.,"Total orders: 4; Total revenue: £1,797.24; Total units: 2,341; Unique products: 22; Recency: 75 days; Average order value: £449.31; Revenue segment: Very High; Activity segment: Recently Inactive.","This customer belongs to the Very High revenue segment and is classified as Recently Inactive. They have placed 4 orders with £1,797.24 in total revenue. The customer purchased 22 unique products and has a recency of 75 days."


In [7]:
print("Revenue segment distribution:")
print(
    customer_features["Revenue_Segment"]
    .value_counts()
)

print("\nActivity segment distribution:")
print(
    customer_features["Activity_Segment"]
    .value_counts()
)

Revenue segment distribution:
Revenue_Segment
Very High    1084
Medium       1084
Low          1084
High         1083
Name: count, dtype: int64

Activity segment distribution:
Activity_Segment
Active               1647
Recently Inactive    1240
Inactive              862
At Risk               586
Name: count, dtype: int64


In [8]:
from sklearn.model_selection import train_test_split

train_df, validation_df = train_test_split(
    instruction_df,
    test_size=0.2,
    random_state=42
)

train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)

print("Training examples:", len(train_df))
print("Validation examples:", len(validation_df))

Training examples: 3468
Validation examples: 867


In [9]:
train_df.to_json(
    "../data/finetuning_train.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

validation_df.to_json(
    "../data/finetuning_validation.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

print("Saved:")
print("../data/finetuning_train.jsonl")
print("../data/finetuning_validation.jsonl")

Saved:
../data/finetuning_train.jsonl
../data/finetuning_validation.jsonl


## Baseline Model

In [10]:
%pip install datasets peft accelerate trl

  Using cached fsspec-2026.6.0-py3-none-any.whl.metadata (10 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached frozenlist-1.8.0-cp311-cp311-win_amd64.whl.metadata (21 kB)
  Using cached propcache-0.5.2-cp311-cp311-win_amd64.whl.metadata (17 kB)
   ---------------------------------------- 0.0/559.1 kB ? eta -:--:--
   ---------------------------------------- 559.1/559.1 kB 6.3 MB/s  0:00:00
Using cached fsspec-2026.6.0-py3-none-any.whl (203 kB)
   ---------------------------------------- 0.0/832.9 kB ? eta -:--:--
   ---------------------------------------- 832.9/832.9 kB 12.4 MB/s  0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 16.2 MB/s  0:00:00
Using cached aiosignal-1.4.0-py3-none-any.whl (7.5 kB)
Using cached frozenlist-1.8.0-cp311-cp311-win_amd64.whl (44 kB)
Using cached propcache-0.5.2-cp311-cp311-win_amd64.whl (42 kB)
   ----------------------------------------

In [13]:
import datasets
import peft
import accelerate
import trl

print("datasets:", datasets.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)
print("trl:", trl.__version__)

datasets: 5.0.1
peft: 0.21.0
accelerate: 1.15.0
trl: 1.13.0


In [14]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

baseline_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto"
)

print("Model loaded:", model_name)
print("Parameters:",
    f"{sum(p.numel() for p in baseline_model.parameters()):,}")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

e:\commerceiq\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sadiya Sajid\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-0.5B-Instruct
Parameters: 494,032,768


In [15]:
def generate_baseline(instruction, user_input, max_new_tokens=120):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a CommerceIQ business analytics assistant. "
                "Use only the information provided by the user."
            )
        },
        {
            "role": "user",
            "content": f"{instruction}\n\n{user_input}"
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = baseline_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

In [16]:
sample = validation_df.iloc[0]

baseline_output = generate_baseline(
    sample["instruction"],
    sample["input"]
)

print("INPUT:")
print(sample["input"])

print("\nEXPECTED OUTPUT:")
print(sample["output"])

print("\nBASELINE MODEL OUTPUT:")
print(baseline_output)

INPUT:
Total orders: 2; Total revenue: £2,222.21; Total units: 1,654; Unique products: 92; Recency: 168 days; Average order value: £1,111.11; Revenue segment: Very High; Activity segment: At Risk.

EXPECTED OUTPUT:
This customer belongs to the Very High revenue segment and is classified as At Risk. They have placed 2 orders with £2,222.21 in total revenue. The customer purchased 92 unique products and has a recency of 168 days.

BASELINE MODEL OUTPUT:
**Business Interpretation:**

The analysis reveals that the customer has been consistently making purchases over the past 168 days with an average order value of £1,111.11. This indicates a high level of repeat business and consistent spending habits. The total number of unique products ordered is 92, suggesting a diverse range of items being purchased regularly. The recency (168 days) suggests that the customer may be new to the brand or experiencing a period of increased activity due to recent promotions or changes in their shopping hab

## LoRA Configuration

In [17]:
import torch

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

print("PEFT imported successfully.")

PEFT imported successfully.


In [18]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    bias="none"
)

print(lora_config)

LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.21.0', base_model_name_or_path=None, revision=None, inference_mode=False, r=8, target_modules={'q_proj', 'o_proj', 'k_proj', 'v_proj'}, exclude_modules=None, lora_alpha=16, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, kasa_config=None, ensure_weight_tying=False)


In [19]:
lora_model = get_peft_model(
    baseline_model,
    lora_config
)

lora_model.print_trainable_parameters()

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


In [20]:
total_params = sum(
    p.numel()
    for p in lora_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in lora_model.parameters()
    if p.requires_grad
)

print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")
print(
    "Trainable percentage:",
    f"{100 * trainable_params / total_params:.4f}%"
)

Total parameters: 495,114,112
Trainable parameters: 1,081,344
Trainable percentage: 0.2184%


## LoRA Fine-tuning

In [21]:
def format_example(row):
    return (
        "### Instruction:\n"
        f"{row['instruction']}\n\n"
        "### Customer Data:\n"
        f"{row['input']}\n\n"
        "### Business Interpretation:\n"
        f"{row['output']}"
    )

train_sample = train_df.sample(
    n=300,
    random_state=42
).reset_index(drop=True)

validation_sample = validation_df.sample(
    n=50,
    random_state=42
).reset_index(drop=True)

train_texts = [
    format_example(row)
    for _, row in train_sample.iterrows()
]

validation_texts = [
    format_example(row)
    for _, row in validation_sample.iterrows()
]

print("Training examples:", len(train_texts))
print("Validation examples:", len(validation_texts))

print("\nExample:")
print(train_texts[0])

Training examples: 300
Validation examples: 50

Example:
### Instruction:
Analyze this customer's purchasing behavior and provide a concise business interpretation.

### Customer Data:
Total orders: 4; Total revenue: £1,932.23; Total units: 1,089; Unique products: 35; Recency: 8 days; Average order value: £483.06; Revenue segment: Very High; Activity segment: Active.

### Business Interpretation:
This customer belongs to the Very High revenue segment and is classified as Active. They have placed 4 orders with £1,932.23 in total revenue. The customer purchased 35 unique products and has a recency of 8 days.


In [22]:
from datasets import Dataset

train_dataset = Dataset.from_dict({
    "text": train_texts
})

validation_dataset = Dataset.from_dict({
    "text": validation_texts
})

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=256
    )

tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

tokenized_validation = validation_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

print("Tokenized training examples:", len(tokenized_train))
print("Tokenized validation examples:", len(tokenized_validation))

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Tokenized training examples: 300
Tokenized validation examples: 50


In [5]:
import torch
import transformers
import accelerate
import peft
import trl

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("CUDA:", torch.cuda.is_available())

Python: e:\commerceiq\.venv\Scripts\python.exe
PyTorch: 2.14.0+cpu
Transformers: 5.17.0
Accelerate: 1.15.0
PEFT: 0.21.0
TRL: 1.13.0
CUDA: False


In [2]:
import sys
import subprocess

# Install required packages into the current notebook environment
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "transformers==5.17.0",
    "datasets==5.0.1",
    "peft==0.21.0",
    "accelerate==1.15.0"
])

import torch
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

print("Setup imports loaded.")
print("Python:", sys.executable)
print("Transformers:", __import__("transformers").__version__)
print("PEFT:", __import__("peft").__version__)

Setup imports loaded.
Python: e:\commerceiq\.venv\Scripts\python.exe
Transformers: 5.17.0
PEFT: 0.21.0


In [3]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

baseline_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto"
)

print("Model loaded.")
print("Parameters:",
    f"{sum(p.numel() for p in baseline_model.parameters()):,}")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded.
Parameters: 494,032,768


In [4]:
train_df = pd.read_json(
    "../data/finetuning_train.jsonl",
    lines=True
)

validation_df = pd.read_json(
    "../data/finetuning_validation.jsonl",
    lines=True
)

print("Training examples:", len(train_df))
print("Validation examples:", len(validation_df))

Training examples: 3468
Validation examples: 867


In [5]:
train_sample = train_df.sample(
    n=300,
    random_state=42
).reset_index(drop=True)

validation_sample = validation_df.sample(
    n=50,
    random_state=42
).reset_index(drop=True)

def format_example(row):
    return (
        "### Instruction:\n"
        f"{row['instruction']}\n\n"
        "### Customer Data:\n"
        f"{row['input']}\n\n"
        "### Business Interpretation:\n"
        f"{row['output']}"
    )

train_texts = [
    format_example(row)
    for _, row in train_sample.iterrows()
]

validation_texts = [
    format_example(row)
    for _, row in validation_sample.iterrows()
]

print("CPU training examples:", len(train_texts))
print("CPU validation examples:", len(validation_texts))

CPU training examples: 300
CPU validation examples: 50


In [6]:
train_dataset = Dataset.from_dict({
    "text": train_texts
})

validation_dataset = Dataset.from_dict({
    "text": validation_texts
})

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=256
    )

tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

tokenized_validation = validation_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

print("Tokenized training:", len(tokenized_train))
print("Tokenized validation:", len(tokenized_validation))

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Tokenized training: 300
Tokenized validation: 50


In [7]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    bias="none"
)

lora_model = get_peft_model(
    baseline_model,
    lora_config
)

lora_model.print_trainable_parameters()

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


In [8]:
from torch.utils.data import DataLoader
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

train_loader = DataLoader(
    tokenized_train,
    batch_size=1,
    shuffle=True,
    collate_fn=data_collator
)

validation_loader = DataLoader(
    tokenized_validation,
    batch_size=1,
    shuffle=False,
    collate_fn=data_collator
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))

Training batches: 300
Validation batches: 50


In [9]:
device = torch.device("cpu")
lora_model = lora_model.to(device)

optimizer = torch.optim.AdamW(
    lora_model.parameters(),
    lr=2e-4
)

print("Device:", device)
print("Optimizer:", type(optimizer).__name__)
print("Learning rate:", 2e-4)

Device: cpu
Optimizer: AdamW
Learning rate: 0.0002


In [10]:
import time

lora_model.train()

max_steps = 20
training_losses = []

start_time = time.time()

for step, batch in enumerate(train_loader):
    if step >= max_steps:
        break

    batch = {
        key: value.to(device)
        for key, value in batch.items()
    }

    optimizer.zero_grad()

    outputs = lora_model(**batch)
    loss = outputs.loss

    loss.backward()
    optimizer.step()

    training_losses.append(loss.item())

    if (step + 1) % 5 == 0:
        print(
            f"Step {step + 1}/{max_steps} | "
            f"Loss: {loss.item():.4f}"
        )

elapsed = time.time() - start_time

print("\nTraining complete.")
print("Steps:", len(training_losses))
print("Initial loss:", f"{training_losses[0]:.4f}")
print("Final loss:", f"{training_losses[-1]:.4f}")
print("Time:", f"{elapsed / 60:.2f} minutes")

Step 5/20 | Loss: 1.8735
Step 10/20 | Loss: 1.4028
Step 15/20 | Loss: 0.9184
Step 20/20 | Loss: 0.4546

Training complete.
Steps: 20
Initial loss: 2.2679
Final loss: 0.4546
Time: 2.06 minutes


In [11]:
lora_model.eval()

validation_losses = []

with torch.no_grad():
    for batch in validation_loader:
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        outputs = lora_model(**batch)
        validation_losses.append(outputs.loss.item())

validation_loss = sum(validation_losses) / len(validation_losses)

print("Validation loss:", f"{validation_loss:.4f}")

Validation loss: 0.4963


In [12]:
def generate_lora_response(customer_input):
    prompt = (
        "### Instruction:\n"
        "Analyze this customer's purchasing behavior and provide a concise business interpretation.\n\n"
        "### Customer Data:\n"
        f"{customer_input}\n\n"
        "### Business Interpretation:\n"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = lora_model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return generated


test_customer = validation_sample.iloc[0]["input"]
expected_output = validation_sample.iloc[0]["output"]

print("EXPECTED:")
print(expected_output)

print("\nAFTER LoRA:")
print(generate_lora_response(test_customer))

EXPECTED:
This customer belongs to the Medium revenue segment and is classified as Recently Inactive. They have placed 4 orders with £592.97 in total revenue. The customer purchased 31 unique products and has a recency of 82 days.

AFTER LoRA:
### Instruction:
Analyze this customer's purchasing behavior and provide a concise business interpretation.

### Customer Data:
Total orders: 4; Total revenue: £592.97; Total units: 302; Unique products: 31; Recency: 82 days; Average order value: £148.24; Revenue segment: Medium; Activity segment: Recently Inactive.

### Business Interpretation:
This customer belongs to the Medium revenue segment and is classified as Recently Inactive. They have placed 4 orders with £592.97 in total revenue. The customer purchased 31 unique products and has a recency of 82 days. This customer is considered recently inactive based on their recent purchase activity.


## Before vs After Evaluation

In [13]:

baseline_model.eval()

def generate_model_response(model, customer_input):
    prompt = (
        "### Instruction:\n"
        "Analyze this customer's purchasing behavior and provide a concise business interpretation.\n\n"
        "### Customer Data:\n"
        f"{customer_input}\n\n"
        "### Business Interpretation:\n"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )


evaluation_rows = []

for _, row in validation_sample.iterrows():
    baseline_output = generate_model_response(
        baseline_model,
        row["input"]
    )

    lora_output = generate_model_response(
        lora_model,
        row["input"]
    )

    evaluation_rows.append({
        "input": row["input"],
        "expected": row["output"],
        "baseline": baseline_output,
        "lora": lora_output
    })

evaluation_df = pd.DataFrame(evaluation_rows)

print("Evaluation examples:", len(evaluation_df))

Evaluation examples: 50


In [14]:
import re

def extract_expected_facts(text):
    revenue_match = re.search(r"£[\d,]+\.\d{2}", text)
    orders_match = re.search(r"(\d+) orders", text)
    products_match = re.search(r"(\d+) unique products", text)
    recency_match = re.search(r"recency of (\d+) days", text)

    segments = re.findall(
        r"(Very High|High|Medium|Low)",
        text
    )

    activity = re.findall(
        r"(Active|Recently Inactive|At Risk|Inactive)",
        text
    )

    return {
        "revenue": revenue_match.group(0) if revenue_match else None,
        "orders": orders_match.group(1) if orders_match else None,
        "products": products_match.group(1) if products_match else None,
        "recency": recency_match.group(1) if recency_match else None,
        "revenue_segment": segments[0] if segments else None,
        "activity_segment": activity[0] if activity else None
    }


def factual_match(expected, generated):
    expected_facts = extract_expected_facts(expected)
    generated_facts = extract_expected_facts(generated)

    matches = []

    for key in expected_facts:
        matches.append(
            expected_facts[key] == generated_facts[key]
        )

    return sum(matches) / len(matches)


evaluation_df["Baseline_Factual_Score"] = evaluation_df.apply(
    lambda row: factual_match(row["expected"], row["baseline"]),
    axis=1
)

evaluation_df["LoRA_Factual_Score"] = evaluation_df.apply(
    lambda row: factual_match(row["expected"], row["lora"]),
    axis=1
)

print(
    "Baseline factual score:",
    f"{evaluation_df['Baseline_Factual_Score'].mean():.3f}"
)

print(
    "LoRA factual score:",
    f"{evaluation_df['LoRA_Factual_Score'].mean():.3f}"
)

Baseline factual score: 0.960
LoRA factual score: 0.960


In [15]:
baseline_score = evaluation_df["Baseline_Factual_Score"].mean()
lora_score = evaluation_df["LoRA_Factual_Score"].mean()

comparison = pd.DataFrame({
    "Model": ["Baseline", "LoRA"],
    "Factual_Score": [baseline_score, lora_score]
})

display(comparison)

print(
    f"Improvement: {(lora_score - baseline_score):.3f}"
)

,Model,Factual_Score
0,Baseline,0.96
1,LoRA,0.96


Improvement: 0.000


In [16]:
evaluation_df.to_csv(
    "../data/finetuning_evaluation.csv",
    index=False
)

comparison.to_csv(
    "../data/finetuning_comparison.csv",
    index=False
)

print("Saved:")
print("../data/finetuning_evaluation.csv")
print("../data/finetuning_comparison.csv")

Saved:
../data/finetuning_evaluation.csv
../data/finetuning_comparison.csv


## QLoRA — Quantized LoRA

In [17]:
print("QLoRA concept:")
print()
print("LoRA:")
print("- Base model remains frozen")
print("- Small trainable adapter matrices are added")
print("- Only adapter parameters are updated")
print()
print("QLoRA:")
print("- Same LoRA adapter approach")
print("- Base model weights are loaded in low precision")
print("- Commonly 4-bit quantization")
print("- Reduces GPU memory requirements")
print("- Enables larger models to be fine-tuned on smaller GPUs")
print()
print("CommerceIQ local environment:")
print("- Device: CPU")
print("- CUDA available:", torch.cuda.is_available())
print()
print("Therefore, full QLoRA training is not practical in the current local CPU environment.")

QLoRA concept:

LoRA:
- Base model remains frozen
- Small trainable adapter matrices are added
- Only adapter parameters are updated

QLoRA:
- Same LoRA adapter approach
- Base model weights are loaded in low precision
- Commonly 4-bit quantization
- Reduces GPU memory requirements
- Enables larger models to be fine-tuned on smaller GPUs

CommerceIQ local environment:
- Device: CPU
- CUDA available: False

Therefore, full QLoRA training is not practical in the current local CPU environment.


In [18]:
qlora_config = {
    "quantization": "4-bit",
    "compute_dtype": "float16/bfloat16",
    "quant_type": "NF4",
    "double_quantization": True,
    "adapter": "LoRA",
    "rank": 8,
    "alpha": 16,
    "dropout": 0.05
}

print("Planned QLoRA configuration:")
for key, value in qlora_config.items():
    print(f"{key}: {value}")

Planned QLoRA configuration:
quantization: 4-bit
compute_dtype: float16/bfloat16
quant_type: NF4
double_quantization: True
adapter: LoRA
rank: 8
alpha: 16
dropout: 0.05


In [19]:
print("Hardware check")
print("----------------")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU: None")
    print()
    print("QLoRA execution will be documented but not run locally.")

Hardware check
----------------
PyTorch: 2.14.0+cpu
CUDA available: False
GPU: None

QLoRA execution will be documented but not run locally.


## Create Chosen and Rejected Responses

In [21]:
def create_preference_example(row):
    chosen = row["output"]

    rejected = (
        f"This customer is a high-value customer who appears to be "
        f"losing interest. They may have stopped purchasing because of "
        f"pricing changes or competing brands. A targeted promotion "
        f"could help bring them back."
    )

    return {
        "prompt": row["instruction"] + "\n\n" + row["input"],
        "chosen": chosen,
        "rejected": rejected
    }


preference_data = [
    create_preference_example(row)
    for _, row in validation_sample.iterrows()
]

preference_df = pd.DataFrame(preference_data)

print("Preference examples:", len(preference_df))
print("Columns:", preference_df.columns.tolist())

Preference examples: 50
Columns: ['prompt', 'chosen', 'rejected']


In [22]:
print("PROMPT:")
print(preference_df.iloc[0]["prompt"])

print("\nCHOSEN:")
print(preference_df.iloc[0]["chosen"])

print("\nREJECTED:")
print(preference_df.iloc[0]["rejected"])

PROMPT:
Analyze this customer's purchasing behavior and provide a concise business interpretation.

Total orders: 4; Total revenue: £592.97; Total units: 302; Unique products: 31; Recency: 82 days; Average order value: £148.24; Revenue segment: Medium; Activity segment: Recently Inactive.

CHOSEN:
This customer belongs to the Medium revenue segment and is classified as Recently Inactive. They have placed 4 orders with £592.97 in total revenue. The customer purchased 31 unique products and has a recency of 82 days.

REJECTED:
This customer is a high-value customer who appears to be losing interest. They may have stopped purchasing because of pricing changes or competing brands. A targeted promotion could help bring them back.


## Preference Data Inspection

In [23]:
print("Preference dataset shape:", preference_df.shape)

print("\nMissing values:")
print(preference_df.isnull().sum())

print("\nAverage character lengths:")
print(
    "Prompt:",
    round(preference_df["prompt"].str.len().mean(), 1)
)

print(
    "Chosen:",
    round(preference_df["chosen"].str.len().mean(), 1)
)

print(
    "Rejected:",
    round(preference_df["rejected"].str.len().mean(), 1)
)

Preference dataset shape: (50, 3)

Missing values:
prompt      0
chosen      0
rejected    0
dtype: int64

Average character lengths:
Prompt: 275.6
Chosen: 214.1
Rejected: 203.0


In [24]:
preference_df.to_json(
    "../data/dpo_preference_dataset.jsonl",
    orient="records",
    lines=True
)

print("Saved:")
print("../data/dpo_preference_dataset.jsonl")

Saved:
../data/dpo_preference_dataset.jsonl


## DPO Training Configuration

In [25]:
dpo_config = {
    "base_model": "Qwen/Qwen2.5-0.5B-Instruct",
    "preference_method": "DPO",
    "beta": 0.1,
    "learning_rate": 5e-5,
    "max_length": 512,
    "max_prompt_length": 256,
    "batch_size": 1,
    "gradient_accumulation": 4
}

print("DPO configuration")
print("-----------------")

for key, value in dpo_config.items():
    print(f"{key}: {value}")

DPO configuration
-----------------
base_model: Qwen/Qwen2.5-0.5B-Instruct
preference_method: DPO
beta: 0.1
learning_rate: 5e-05
max_length: 512
max_prompt_length: 256
batch_size: 1
gradient_accumulation: 4


In [27]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "trl==1.13.0"
])

import trl

print("TRL version:", trl.__version__)

try:
    from trl import DPOTrainer
    print("DPOTrainer: available")
except ImportError as e:
    print("DPOTrainer: not available")
    print("Error:", e)

TRL version: 1.13.0
DPOTrainer: available


In [28]:
print("DPO execution decision")
print("----------------------")
print()
print("Local device: CPU")
print("Base model: Qwen/Qwen2.5-0.5B-Instruct")
print("Preference examples:", len(preference_df))
print()
print("Decision:")
print("DPO is configured and the preference dataset is prepared,")
print("but full DPO training is not executed locally.")
print()
print("Reason:")
print("DPO requires multiple model forward passes per preference")
print("pair and is substantially more expensive than the short")
print("LoRA demonstration already completed on CPU.")

DPO execution decision
----------------------

Local device: CPU
Base model: Qwen/Qwen2.5-0.5B-Instruct
Preference examples: 50

Decision:
DPO is configured and the preference dataset is prepared,
but full DPO training is not executed locally.

Reason:
DPO requires multiple model forward passes per preference
pair and is substantially more expensive than the short
LoRA demonstration already completed on CPU.
